<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/03_redes_fundamentos/32_gradiente_y_backpropagation.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Descenso de gradiente y backpropagation

**Pregunta guía:** ¿Cómo atribuye la regla de la cadena cada error a cada parámetro?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Gradiente

Para parámetros $\theta$, descenso de gradiente actualiza
$\theta_{t+1}=\theta_t-\eta\nabla_\theta L$. Backpropagation no es un
optimizador: es una forma eficiente de calcular productos de derivadas
desde la salida hacia las entradas mediante la regla de la cadena.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Grafo escalar: L=(wx+b-y)^2
x, y, w, b = 2.0, 5.0, 0.7, -0.2
pred = w * x + b
pérdida = (pred - y) ** 2
dL_dpred = 2 * (pred - y)
dL_dw = dL_dpred * x
dL_db = dL_dpred
print({"pred": pred, "L": pérdida, "dL/dw": dL_dw, "dL/db": dL_db})

def L(w_, b_): return (w_ * x + b_ - y) ** 2
h = 1e-6
num_w = (L(w + h, b) - L(w - h, b)) / (2 * h)
num_b = (L(w, b + h) - L(w, b - h)) / (2 * h)
print("error gradiente:", abs(num_w-dL_dw), abs(num_b-dL_db))


## Backpropagation matricial

Para una red de regresión 1→20→1 con `tanh` y MSE:
$\delta^{(2)}=2(\hat y-y)/N$,
$\nabla W^{(2)}=(a^{(1)})^T\delta^{(2)}$ y
$\delta^{(1)}=(\delta^{(2)}(W^{(2)})^T)\odot(1-\tanh^2 z^{(1)})$.
Guardar $z$ y $a$ durante forward evita recalcularlos.


In [ ]:
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)
X = np.linspace(-2.5, 2.5, 240)[:, None]
y = np.sin(2 * X) + 0.15 * X
W1 = rng.normal(0, np.sqrt(1), (1, 20)); b1 = np.zeros((1, 20))
W2 = rng.normal(0, np.sqrt(1/20), (20, 1)); b2 = np.zeros((1, 1))

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = np.tanh(z1)
    pred = a1 @ W2 + b2
    return z1, a1, pred

def gradientes(X, y, W1, b1, W2, b2):
    z1, a1, pred = forward(X, W1, b1, W2, b2)
    n = len(X)
    d_pred = 2 * (pred - y) / n
    dW2 = a1.T @ d_pred
    db2 = d_pred.sum(axis=0, keepdims=True)
    dz1 = (d_pred @ W2.T) * (1 - np.tanh(z1) ** 2)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)
    return (dW1, db1, dW2, db2), np.mean((pred-y)**2)


In [ ]:
# Gradient check de cinco entradas aleatorias de W1.
analíticos, _ = gradientes(X, y, W1, b1, W2, b2)
dW1 = analíticos[0]
h = 1e-5
for j in [0, 3, 7, 12, 19]:
    original = W1[0, j]
    W1[0, j] = original + h
    L_plus = np.mean((forward(X, W1, b1, W2, b2)[2] - y) ** 2)
    W1[0, j] = original - h
    L_minus = np.mean((forward(X, W1, b1, W2, b2)[2] - y) ** 2)
    W1[0, j] = original
    numérico = (L_plus - L_minus) / (2*h)
    print(j, "analítico=", dW1[0,j], "numérico=", numérico)


In [ ]:
historial = []
lr = 0.03
for época in range(2_000):
    (dW1, db1, dW2, db2), mse = gradientes(X, y, W1, b1, W2, b2)
    W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2
    if época % 50 == 0: historial.append(mse)

pred = forward(X, W1, b1, W2, b2)[2]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(len(historial))*50, historial)
axes[0].set(yscale="log", xlabel="época", ylabel="MSE")
axes[1].plot(X, y, "k--", label="objetivo")
axes[1].plot(X, pred, label="red")
axes[1].legend()
plt.show()


**Ejercicios:** derive gradientes de $b_1$; pruebe tasas 0.001, 0.03 y
0.5; grafique normas de gradiente; sustituya `tanh` por su activación del
notebook anterior; explique la diferencia entre derivación automática y
diferenciación numérica.
